# spVIPES Disentanglement Ablation Walkthrough

This notebook demonstrates the **disentanglement objective** added to spVIPES
(inspired by CellDISECT's cross-covariate decoupling MLPs and
Multi-ContrastiveVAE) and shows how to run systematic ablations.

**Note on naming:** these losses are *not* the original Mutual Information
Gap metric (Chen et al. 2018), which is a post-hoc disentanglement metric.
What we implement is a mix of **adversarial domain-invariance losses**
(DANN-style, via gradient reversal) and **supervised classification losses**
acting as variational MI lower bounds, plus an optional **prototype
contrastive InfoNCE** on the shared latent.

## Goal

Enforce that each latent space encodes only the factor it 'owns':

* `z_shared`  : informative about cell-type label, uninformative about group
* `z_private` : informative about group, uninformative about cell-type label

Realised through 4 auxiliary classifiers + 1 optional contrastive term:

| Component | Latent | Target | Encoder direction | Mechanism | Needs labels? |
|---|---|---|---|---|---|
| `q_label_shared` | z_shared | label | preserve | supervised CE (MI lower bound) | yes |
| `q_group_shared` | z_shared | group | erase | adversarial CE via GRL | no |
| `q_group_private` | z_private | group | preserve | supervised CE | no |
| `q_label_private` | z_private | label | erase | adversarial CE via GRL | yes |
| Contrastive | z_shared | label | preserve | InfoNCE on EMA prototypes | yes |

## What this notebook does

1. Loads the DIALOGUE example dataset (3 groups, labelled cells).
2. Trains spVIPES with each `disentangle_preset` and reports metrics.
3. Runs single-component ablations starting from `disentangle_preset='full'`.
4. Compares ablations on group-mixing (shared) and label-preservation (shared).

Each component can be independently disabled by setting its weight to 0,
or activated via a named preset.

## Requirements & Compatibility

> **scvi-tools ≥1.0 required.** spVIPES targets scvi-tools 1.x and `lightning.pytorch` (was 0.20.x + `pytorch_lightning`). The deprecated `use_gpu=True` kwarg on `model.train(...)` has been **removed**; pass `accelerator="gpu", devices=1` (or `"auto"`) inside `**trainer_kwargs` instead. Minimum Python is 3.10. Several private scvi-tools modules removed in 1.x are now vendored under `spVIPES.data`. See `CHANGELOG.md` for full details.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import scanpy as sc
import torch
import matplotlib.pyplot as plt

import pertpy as pt
import spVIPES
from spVIPES.model._disentangle_presets import DISENTANGLE_PRESETS

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
sc.settings.set_figure_params(dpi=100, frameon=False)

# Inspect the available presets
for name, weights in DISENTANGLE_PRESETS.items():
    print(f"{name}: {weights}")

## 1. Load and prepare the dataset

We use the DIALOGUE example: PBMCs annotated by cell type (`cell.subtypes`)
and clinical status (`clinical.status`). Groups are defined by
`clinical.status`; labels for label-based PoE are `cell.subtypes`.

In [ ]:
adata_full = pt.dt.dialogue_example()
sc.pp.normalize_total(adata_full)
sc.pp.log1p(adata_full)

# Split by clinical status into N groups
groups_dict = {
    str(s): adata_full[adata_full.obs["clinical.status"] == s].copy()
    for s in adata_full.obs["clinical.status"].unique()
}
for s, a in groups_dict.items():
    print(f"  {s}: {a.n_obs} cells")

adata = spVIPES.data.prepare_adatas(groups_dict)
spVIPES.model.spVIPES.setup_anndata(
    adata, groups_key="groups", label_key="cell.subtypes"
)

group_indices_list = [
    np.where(adata.obs["groups"] == g)[0]
    for g in adata.obs["groups"].cat.categories
]


## 2. Helper: train a model with a given disentanglement configuration

In [ ]:
def train_and_score(adata, group_indices_list, *, max_epochs=50, **disentangle_kwargs):
    """Train a spVIPES model with given disentanglement kwargs and return latent + history."""
    model = spVIPES.model.spVIPES(
        adata,
        n_dimensions_shared=20,
        n_dimensions_private=8,
        **disentangle_kwargs,
    )
    model.train(group_indices_list, max_epochs=max_epochs, batch_size=256)
    z = model.get_latent_representation(group_indices_list)
    return model, z


## 3. Compare presets

We train one model per preset and tabulate group mixing (kBET-like) vs.
label preservation (k-NN purity) on the **shared** latent.

In [ ]:
results = {}
for preset in ["off", "shared_only", "private_only", "adversarial_only",
               "supervised_only", "no_contrastive", "full"]:
    print(f"\n=== preset={preset} ===")
    model, z = train_and_score(adata, group_indices_list, disentangle_preset=preset)
    spVIPES.utils.store_latents(adata, z, group_indices_list)
    z_shared = adata.obsm["X_spVIPES_shared"]
    groups = adata.obs["groups"].values
    labels = adata.obs["cell.subtypes"].values
    results[preset] = {
        "group_mixing_shared": spVIPES.metrics.kbet(z_shared, groups),
        "label_purity_shared": spVIPES.metrics.knn_purity(z_shared, labels),
    }

preset_df = pd.DataFrame(results).T
preset_df


## 4. Single-component ablations (starting from `full`)

For each component, set its weight to 0 and re-train. The drop in
metrics tells you what that component contributes.

In [ ]:
ablate_components = [
    "disentangle_group_shared_weight",
    "disentangle_label_shared_weight",
    "disentangle_group_private_weight",
    "disentangle_label_private_weight",
    "contrastive_weight",
]

ablation_results = {}
for component in ablate_components:
    label = f"full minus {component}"
    print(f"\n=== {label} ===")
    model, z = train_and_score(
        adata, group_indices_list, disentangle_preset="full", **{component: 0.0}
    )
    spVIPES.utils.store_latents(adata, z, group_indices_list)
    z_shared = adata.obsm["X_spVIPES_shared"]
    groups = adata.obs["groups"].values
    labels = adata.obs["cell.subtypes"].values
    ablation_results[label] = {
        "group_mixing_shared": spVIPES.metrics.kbet(z_shared, groups),
        "label_purity_shared": spVIPES.metrics.knn_purity(z_shared, labels),
    }

ablation_df = pd.DataFrame(ablation_results).T
ablation_df


## 5. Visual summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

preset_df.plot.bar(ax=axes[0])
axes[0].set_title("Disentanglement presets: shared-latent metrics")
axes[0].set_ylabel("score (higher is better)")
axes[0].tick_params(axis="x", rotation=30)
axes[0].legend()

ablation_df.plot.bar(ax=axes[1])
axes[1].set_title("Single-component ablations from full")
axes[1].set_ylabel("score (higher is better)")
axes[1].tick_params(axis="x", rotation=30)
axes[1].legend()

plt.tight_layout()
plt.show()

## 6. Interpreting the results

**Expected patterns:**

- `disentangle_preset='full'` should produce the highest combined score: high group
  mixing in z_shared (`q_group_shared` erases group identity) plus high label
  preservation (`q_label_shared` + contrastive enforce cell-type clustering).
- Removing `disentangle_group_shared_weight` should drop **group mixing** the most.
- Removing `disentangle_label_shared_weight` or `contrastive_weight` should drop
  **label purity** the most.
- `private_only` should leave the shared latent unchanged from `off` since
  it only constrains the private latents.

## 7. Custom configurations

Combine a preset with overrides for any custom configuration:

```python
# Strong adversarial group erasure with weak label supervision
model = spVIPES.model.spVIPES(
    adata,
    disentangle_preset="full",
    disentangle_group_shared_weight=2.0,    # boost adversarial
    disentangle_label_shared_weight=0.5,    # weaken supervision
    contrastive_weight=0.0,         # disable contrastive
)
```

**Validation:** if labels are not registered, setting `disentangle_label_*_weight > 0`
or `contrastive_weight > 0` raises a clear `ValueError` at construction time.
Group classifiers (`q_group_shared`, `q_group_private`) work without labels.